# Pandas — Data Analysis with DataFrames

## What is pandas and why use it?
Pandas provides two core data structures — `Series` (1D labelled array) and `DataFrame` (2D table with labelled rows and columns). It makes loading, cleaning, transforming, and analysing tabular data fast and expressive.

If you work with CSVs, databases, Excel files, or any tabular data, pandas is the tool.

```bash
pip install pandas
```

## Table of Contents
1. Series vs DataFrame
2. Creating DataFrames
3. Reading files
4. Exploring data: head/tail/info/describe
5. Selecting data: `[]`, `loc`, `iloc`
6. Filtering: boolean masks and `query()`
7. Adding and dropping columns
8. Handling missing data
9. Sorting
10. `groupby` + aggregation
11. `merge` and `concat`
12. `apply`, `map`
13. `value_counts`, string ops, datetime ops
14. Exporting data
15. Common Mistakes
16. Exercises + Solutions
17. Mini-Project: Sales Data Analysis

---
## 1. Series vs DataFrame

In [ ]:
import pandas as pd
import numpy as np

# Series — a 1D array with a labelled index
temps = pd.Series([22.1, 19.8, 25.4, 23.0, 18.7],
                  index=["Mon", "Tue", "Wed", "Thu", "Fri"],
                  name="temperature")
print("Series:\n", temps)
print("\nMean:", temps.mean())
print("Wed temp:", temps["Wed"])

# DataFrame — a 2D table; think of it as a dict of Series
df = pd.DataFrame({
    "name":  ["Alice", "Bob", "Carol", "Dave"],
    "age":   [25, 32, 28, 45],
    "score": [88.5, 76.0, 92.3, 81.7]
})
print("\nDataFrame:\n", df)

---
## 2. Creating DataFrames

In [ ]:
import pandas as pd

# From list of dicts (each dict = one row)
records = [
    {"product": "Widget", "price": 9.99,  "qty": 100},
    {"product": "Gadget", "price": 24.99, "qty": 50},
    {"product": "Doohickey", "price": 4.99, "qty": 200},
]
df = pd.DataFrame(records)
print("From list of dicts:\n", df)

# Custom index
df.index = ["A", "B", "C"]
print("\nCustom index:\n", df)

# From a numpy array
import numpy as np
arr = np.random.default_rng(0).integers(50, 100, size=(3, 4))
df2 = pd.DataFrame(arr, columns=["Q1", "Q2", "Q3", "Q4"])
print("\nFrom numpy:\n", df2)

---
## 3. Reading Files

In [ ]:
import pandas as pd
from pathlib import Path
import io

# Create a sample CSV in memory for demonstration
csv_data = """date,product,category,sales,units
2024-01-05,Widget A,Electronics,1200.50,45
2024-01-07,Widget B,Electronics,980.00,32
2024-01-10,Chair,Furniture,3200.00,8
2024-01-12,Desk,Furniture,5400.00,6
2024-02-01,Widget A,Electronics,1450.75,55
2024-02-03,Lamp,Furniture,890.25,22
2024-02-08,Widget C,Electronics,,15
2024-02-15,Sofa,Furniture,7800.00,
2024-03-01,Widget A,Electronics,1600.00,60
2024-03-05,Bookshelf,Furniture,2100.00,14
"""

# In practice: pd.read_csv("path/to/file.csv")
df = pd.read_csv(io.StringIO(csv_data), parse_dates=["date"])
print(df)

# Save for later
Path("temp").mkdir(exist_ok=True)
df.to_csv("temp/sales.csv", index=False)

---
## 4. Exploring Data: head / tail / info / describe

In [ ]:
import pandas as pd

df = pd.read_csv("temp/sales.csv", parse_dates=["date"])

print("First 3 rows:")
print(df.head(3))

print("\nLast 3 rows:")
print(df.tail(3))

print("\nShape:", df.shape)  # (rows, cols)
print("\nColumn dtypes:")
print(df.dtypes)

print("\nInfo (non-null counts + memory):")
df.info()

print("\nNumerical summary:")
print(df.describe())

---
## 5. Selecting Data: `[]`, `loc`, `iloc`

In [ ]:
import pandas as pd

df = pd.read_csv("temp/sales.csv", parse_dates=["date"])

# [] — select one column (returns Series) or multiple (returns DataFrame)
print("Single column:\n", df["product"].values)
print("\nMultiple columns:\n", df[["product", "sales"]].head())

# loc — label-based: df.loc[row_label, col_label]
print("\nloc row 0:", df.loc[0])
print("\nloc rows 0-2, specific cols:")
print(df.loc[0:2, ["product", "category", "sales"]])

# iloc — position-based: df.iloc[row_int, col_int]
print("\niloc [0:3, 1:4]:")
print(df.iloc[0:3, 1:4])

---
## 6. Filtering: Boolean Masks and `query()`

In [ ]:
import pandas as pd

df = pd.read_csv("temp/sales.csv", parse_dates=["date"])

# Boolean mask
electronics = df[df["category"] == "Electronics"]
print("Electronics:\n", electronics[["product", "sales"]])

# Multiple conditions — use & (and) | (or), wrap each in ()
big_electronics = df[(df["category"] == "Electronics") & (df["sales"] > 1200)]
print("\nBig electronics sales:")
print(big_electronics[["date", "product", "sales"]])

# query() — more readable string expression
furniture_expensive = df.query("category == 'Furniture' and sales > 3000")
print("\nExpensive furniture (query):")
print(furniture_expensive[["product", "sales"]])

# isin — match against a list
targets = df[df["product"].isin(["Widget A", "Widget B"])]
print("\nWidget A and B:")
print(targets[["date", "product", "sales"]])

---
## 7. Adding and Dropping Columns

In [ ]:
import pandas as pd

df = pd.read_csv("temp/sales.csv", parse_dates=["date"])

# Add a new column
df["revenue_per_unit"] = df["sales"] / df["units"]
df["month"] = df["date"].dt.to_period("M")
print("With new columns:\n", df.head())

# Rename columns
df = df.rename(columns={"sales": "total_sales", "units": "units_sold"})
print("\nRenamed:", df.columns.tolist())

# Drop columns
df_slim = df.drop(columns=["revenue_per_unit", "month"])
print("\nDropped cols:", df_slim.columns.tolist())

# Drop rows
df_clean = df.drop(index=[0, 1])
print("Dropped rows 0,1 — new length:", len(df_clean))

---
## 8. Handling Missing Data

In [ ]:
import pandas as pd

df = pd.read_csv("temp/sales.csv", parse_dates=["date"])

# Detect missing values
print("Null counts per column:")
print(df.isnull().sum())

print("\nRows with any nulls:")
print(df[df.isnull().any(axis=1)])

# Strategy 1: drop rows with any null values
df_dropped = df.dropna()
print(f"\nAfter dropna: {len(df)} → {len(df_dropped)} rows")

# Strategy 2: fill nulls with a value or statistic
df_filled = df.copy()
df_filled["sales"] = df_filled["sales"].fillna(df_filled["sales"].median())
df_filled["units"] = df_filled["units"].fillna(df_filled["units"].median())
print("\nAfter fillna (median):")
print(df_filled[["product", "sales", "units"]])

# Strategy 3: forward-fill (useful for time series)
df_ffill = df.fillna(method="ffill")
print("\nAfter ffill:")
print(df_ffill[["product", "sales", "units"]])

---
## 9. Sorting

In [ ]:
import pandas as pd

df = pd.read_csv("temp/sales.csv", parse_dates=["date"])

# Sort by a single column
by_sales = df.sort_values("sales", ascending=False)
print("By sales desc:\n", by_sales[["product", "sales"]].head())

# Sort by multiple columns
multi = df.sort_values(["category", "sales"], ascending=[True, False])
print("\nBy category asc, then sales desc:")
print(multi[["category", "product", "sales"]])

# Sort by index
df_reindexed = df.set_index("product")
print("\nSorted by index:")
print(df_reindexed.sort_index()[["sales"]].head())

---
## 10. `groupby` + Aggregation

In [ ]:
import pandas as pd

df = pd.read_csv("temp/sales.csv", parse_dates=["date"])

# Simple groupby
by_category = df.groupby("category")["sales"].sum()
print("Total sales by category:\n", by_category)

# Multiple aggregations with agg()
summary = df.groupby("category").agg(
    total_sales=("sales", "sum"),
    avg_sales=("sales", "mean"),
    num_transactions=("sales", "count"),
    max_units=("units", "max")
).round(2)
print("\nSummary by category:")
print(summary)

# Group by multiple columns
df["month"] = df["date"].dt.month
monthly = df.groupby(["month", "category"])["sales"].sum().unstack(fill_value=0)
print("\nMonthly sales by category:")
print(monthly)

---
## 11. `merge` and `concat`

In [ ]:
import pandas as pd

# concat — stack DataFrames vertically or horizontally
jan = pd.DataFrame({"month": ["Jan"], "sales": [5000]})
feb = pd.DataFrame({"month": ["Feb"], "sales": [6200]})
combined = pd.concat([jan, feb], ignore_index=True)
print("concat:\n", combined)

# merge — SQL-style joins
products = pd.DataFrame({
    "product_id": [1, 2, 3],
    "name": ["Widget", "Gadget", "Doohickey"],
    "category": ["Electronics", "Electronics", "Furniture"]
})
orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104],
    "product_id": [1, 3, 1, 2],
    "amount": [250, 80, 250, 150]
})

merged = pd.merge(orders, products, on="product_id", how="left")
print("\nLeft merge (orders + products):")
print(merged)

---
## 12. `apply` and `map`

In [ ]:
import pandas as pd

df = pd.read_csv("temp/sales.csv", parse_dates=["date"])

# map — element-wise transformation on a Series
category_code = {"Electronics": "ELEC", "Furniture": "FURN"}
df["cat_code"] = df["category"].map(category_code)
print("Mapped codes:\n", df[["category", "cat_code"]].head())

# apply on a Series — apply a function to each element
def classify_sale(amount):
    if pd.isna(amount):
        return "unknown"
    elif amount >= 3000:
        return "large"
    elif amount >= 1000:
        return "medium"
    return "small"

df["sale_size"] = df["sales"].apply(classify_sale)
print("\nSale sizes:\n", df[["product", "sales", "sale_size"]])

---
## 13. `value_counts`, String Operations, Datetime Operations

In [ ]:
import pandas as pd

df = pd.read_csv("temp/sales.csv", parse_dates=["date"])

# value_counts — frequency of each value
print("Product counts:")
print(df["product"].value_counts())

# String operations — via .str accessor
df["product_upper"] = df["product"].str.upper()
df["is_widget"] = df["product"].str.startswith("Widget")
df["product_len"] = df["product"].str.len()
print("\nString ops:")
print(df[["product", "product_upper", "is_widget", "product_len"]].head())

# Datetime operations — via .dt accessor
print("\nDatetime ops:")
print("Year: ",  df["date"].dt.year.values)
print("Month:",  df["date"].dt.month.values)
print("DoW:  ",  df["date"].dt.day_name().values)

---
## 14. Exporting Data

In [ ]:
import pandas as pd

df = pd.read_csv("temp/sales.csv", parse_dates=["date"])
summary = df.groupby("category")[["sales", "units"]].sum()

# CSV
summary.to_csv("temp/summary.csv")
print("Saved CSV")

# JSON
summary.to_json("temp/summary.json", orient="records", indent=2)
print("Saved JSON")

# Verify
print("\nLoaded back from CSV:")
print(pd.read_csv("temp/summary.csv"))

---
## 15. Common Mistakes

**`SettingWithCopyWarning`:** Chained assignment like `df[mask]["col"] = val` modifies a copy. Use `df.loc[mask, "col"] = val` instead.

**`loc` vs `iloc`:** `loc` uses *labels* (index values); `iloc` uses *integer positions*. When the index is 0-based integers they look the same — but after filtering the index no longer starts at 0, so `df.loc[0]` and `df.iloc[0]` give different rows.

**`groupby` drops NaN keys by default:** Pass `dropna=False` to `groupby()` to include groups where the key is `NaN`.

**`merge` silently adds `_x` and `_y` suffixes:** When both DataFrames have the same column name (other than the join key), pandas renames them. Check your columns after merging.

**`fillna` does not modify in place by default:** Either assign back (`df["col"] = df["col"].fillna(0)`) or pass `inplace=True`.

---
## 16. Exercises

**Exercise 1:** Load `temp/sales.csv`. Find the product with the highest total sales across all months.

**Exercise 2:** Create a pivot table showing total `sales` for each `category` by `month` (hint: `pd.pivot_table`).

**Exercise 3:** Add a column `sales_rank` that ranks each row by `sales` within its `category` (1 = highest). Use `groupby` + `rank`.

In [ ]:
# Solution 1
import pandas as pd
df = pd.read_csv("temp/sales.csv")
top_product = df.groupby("product")["sales"].sum().idxmax()
top_sales   = df.groupby("product")["sales"].sum().max()
print(f"Top product: {top_product} (${top_sales:.2f})")

In [ ]:
# Solution 2
import pandas as pd
df = pd.read_csv("temp/sales.csv", parse_dates=["date"])
df["month"] = df["date"].dt.to_period("M").astype(str)
pivot = pd.pivot_table(df, values="sales", index="category", columns="month", aggfunc="sum", fill_value=0)
print(pivot)

In [ ]:
# Solution 3
import pandas as pd
df = pd.read_csv("temp/sales.csv")
df["sales_rank"] = df.groupby("category")["sales"].rank(method="dense", ascending=False).astype("Int64")
print(df[["product", "category", "sales", "sales_rank"]].sort_values(["category", "sales_rank"]))

---
## 17. Mini-Project: Sales Data Analysis
Load the sample sales dataset, clean NaN values, group by category, find top products, compute monthly trends, and export a summary.

In [ ]:
import pandas as pd
from pathlib import Path

# --- 1. Load ---
df = pd.read_csv("temp/sales.csv", parse_dates=["date"])
print(f"Loaded {len(df)} rows. Nulls: {df.isnull().sum().sum()}")

# --- 2. Clean ---
df["sales"] = df["sales"].fillna(df.groupby("product")["sales"].transform("median"))
df["units"] = df["units"].fillna(df.groupby("product")["units"].transform("median"))
print(f"After cleaning, nulls: {df.isnull().sum().sum()}")

# --- 3. Derived columns ---
df["month"] = df["date"].dt.to_period("M")
df["revenue_per_unit"] = (df["sales"] / df["units"]).round(2)

# --- 4. Category summary ---
cat_summary = df.groupby("category").agg(
    total_revenue=("sales", "sum"),
    avg_revenue=("sales", "mean"),
    transactions=("sales", "count")
).round(2)
print("\n=== Category Summary ===")
print(cat_summary)

# --- 5. Top products ---
top_products = (
    df.groupby("product")["sales"]
      .sum()
      .sort_values(ascending=False)
      .head(5)
)
print("\n=== Top 5 Products ===")
print(top_products)

# --- 6. Monthly trends ---
monthly = df.groupby(["month", "category"])["sales"].sum().unstack(fill_value=0)
monthly["total"] = monthly.sum(axis=1)
print("\n=== Monthly Trends ===")
print(monthly)

# --- 7. Export ---
with pd.ExcelWriter("temp/sales_report.xlsx") if False else open("/dev/null", "w") as _:
    pass  # Excel export requires openpyxl; skip in demo

cat_summary.to_csv("temp/category_summary.csv")
monthly.to_csv("temp/monthly_trends.csv")
print("\nExported to temp/category_summary.csv and temp/monthly_trends.csv")